In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "real-time-streaming-lakehouse")
dbutils.widgets.text("schema", "ecommerce-events")
dbutils.widgets.text("volume_path", "/Volumes/real-time-streaming-lakehouse/ecommerce-events/ecommerce-events")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME_PATH = dbutils.widgets.get("volume_path")

In [0]:
SILVER_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`silver-events`"
GOLD_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`gold-kpi-metrics`"
GOLD_CHECKPOINT = f"{VOLUME_PATH}/checkpoints/gold"

In [0]:
gold_agg = (
    spark.readStream.table(SILVER_TABLE)
                    .withWatermark("event_timestamp", "10 minutes")
                    .groupBy(F.window(F.col("event_timestamp"), "5 minutes"))
                    .agg(
                        F.count("*").alias("total_events"),
                        F.approx_count_distinct("user_id").alias("unique_users"),
                        F.sum(F.when(F.col("event_type") == "purchase", F.col("amount")).otherwise(F.lit(0.0))).alias("total_revenue"),
                        F.sum(F.when(F.col("event_type") == "purchase", 1)).alias("purchase_count")
                    ).withColumn(
                        "conversion_rate",
                        F.col("purchase_count")/F.col("total_events")
                    ).withColumn(
                        "avg_order_value",
                        F.when(F.col("purchase_count")>0, F.col("total_revenue")/F.col("purchase_count")).otherwise(0)
                    ).withColumn(
                        "window_start",
                        F.col("window.start")
                    ).withColumn(
                        "window_end",
                        F.col("window.end")
                    ).drop("window")
)


In [0]:
query = (
    gold_agg.writeStream.format("delta")
                        .outputMode("complete")
                        .option("checkpointLocation", GOLD_CHECKPOINT)
                        .trigger(availableNow=True)
                        .toTable(GOLD_TABLE)
)

In [0]:
%sql
Select * from `real-time-streaming-lakehouse`.`ecommerce-events`.`gold-kpi-metrics`
order by window_start desc

total_events,unique_users,total_revenue,purchase_count,conversion_rate,avg_order_value,window_start,window_end
48,49,307.09,13,0.2708333333333333,23.62230769230769,2026-06-22T15:15:00.000Z,2026-06-22T15:20:00.000Z
166,162,1006.03,38,0.2289156626506024,26.474473684210526,2026-06-22T15:10:00.000Z,2026-06-22T15:15:00.000Z
86,87,458.28999999999996,19,0.22093023255813954,24.120526315789473,2026-06-22T15:05:00.000Z,2026-06-22T15:10:00.000Z
75,75,1651.06,16,0.21333333333333335,103.19125,2026-06-21T23:50:00.000Z,2026-06-21T23:55:00.000Z
137,141,3917.869999999999,43,0.31386861313868614,91.11325581395346,2026-06-21T23:45:00.000Z,2026-06-21T23:50:00.000Z
88,88,2687.8100000000004,22,0.25,122.17318181818183,2026-06-21T23:40:00.000Z,2026-06-21T23:45:00.000Z


In [0]:
%sql
Select * from `real-time-streaming-lakehouse`.`ecommerce-events`.`gold-kpi-metrics`

total_events,unique_users,total_revenue,purchase_count,conversion_rate,avg_order_value,window_start,window_end
75,75,1651.06,16,0.21333333333333335,103.19125,2026-06-21T23:50:00.000Z,2026-06-21T23:55:00.000Z
88,88,2687.8100000000004,22,0.25,122.17318181818183,2026-06-21T23:40:00.000Z,2026-06-21T23:45:00.000Z
137,141,3917.869999999999,43,0.31386861313868614,91.11325581395346,2026-06-21T23:45:00.000Z,2026-06-21T23:50:00.000Z
